In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
torch.manual_seed(43)
np.random.seed(43)

# Define the path to CSV file
file_path = "/home/oj/Project_code/MLLS Hackaton/compas_propublica.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Preprocess the data - Apply ProPublica filters to remove invalid cases
df = df[(df['days_b_screening_arrest'] <= 30) & (df['days_b_screening_arrest'] >= -30)] 
df = df[df['is_recid'] != -1]
df = df[df['c_charge_degree'] != 'O']
df = df[df['score_text'] != 'N/A']

# Merge Asian and Native American with others in the dataset
df['race'] = df['race'].replace(['Asian', 'Native American'], 'Other')

# Select relevant columns including new features (NOW USING decile_score as label)
df = df[['id', 'sex', 'dob', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 
         'juv_other_count', 'priors_count', 'c_charge_degree', 'c_charge_desc', 
         'decile_score', 'score_text', 'is_recid']].copy()  # Using decile_score as label

# === Feature Engineering & Encoding ===

# Keep original categorical columns for reference
df['sex_original'] = df['sex']
df['race_original'] = df['race']
df['risk_original'] = df['score_text']
df['charge_degree_original'] = df['c_charge_degree']

# --------------------------------------------------------------
#  COMPAS: XGBoost + Influence + SHAP (per obs) + Plots + Excel
# --------------------------------------------------------------
import xgboost as xgb
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
import shap
import warnings; warnings.filterwarnings('ignore')
import os
import torch
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from scipy.stats import mannwhitneyu

# ------------------- 1. Train XGBoost (Classification) -----------------
def train_xgb(X_train, y_train, X_val=None, y_val=None, weights=None):
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=weights)
    dval   = xgb.DMatrix(X_val, label=y_val) if X_val is not None else None
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'eta': 0.05,
        'max_depth': 4,
        'min_child_weight': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 43,
        'tree_method': 'hist'
    }
    evals = [(dval, 'val')] if dval else []
    model = xgb.train(
        params, dtrain, num_boost_round=500,
        evals=evals, early_stopping_rounds=50 if dval else None,
        verbose_eval=False
    )
    return model

# ------------------- 2. Gradient & Hessian (Classification) -----------------
def log_loss_grad_hess(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    grad = y_pred - y_true
    hess = y_pred * (1 - y_pred)
    return grad, hess

def cg_solve(Av_func, b, max_iter=400, tol=1e-6):
    x = np.zeros_like(b); r = b.copy(); p = b.copy(); rs_old = np.dot(r, r)
    for _ in range(max_iter):
        Ap = Av_func(p)
        alpha = rs_old / (np.dot(p, Ap) + 1e-12)
        x = x + alpha * p; r = r - alpha * Ap
        rs_new = np.dot(r, r)
        if np.sqrt(rs_new) < tol: break
        p = r + (rs_new / (rs_old + 1e-12)) * p
        rs_old = rs_new
    return x

# ------------------- 3. Influence Matrix (Classification) -----------------
def compute_influence_matrix_xgb(model, X_train, y_train, X_test, y_test, damping=1.0):
    n_train, n_test = X_train.shape[0], X_test.shape[0]
    dtrain = xgb.DMatrix(X_train); dtest = xgb.DMatrix(X_test)
    prob_train = model.predict(dtrain); prob_test = model.predict(dtest)
    grad_train, hess_train = log_loss_grad_hess(y_train, prob_train)
    grad_test,  hess_test  = log_loss_grad_hess(y_test,  prob_test)

    leaf_idx_train = model.predict(dtrain, pred_leaf=True)
    leaf_idx_test  = model.predict(dtest,  pred_leaf=True)
    n_trees = leaf_idx_train.shape[1]

    leaf_maps = []
    for t in range(n_trees):
        leaves = np.unique(leaf_idx_train[:, t])
        leaf_maps.append({leaf: i for i, leaf in enumerate(leaves)})

    def get_leaf_matrix(leaf_idx):
        rows, cols, data = [], [], []
        offset = 0
        for t in range(n_trees):
            for i in range(leaf_idx.shape[0]):
                leaf = leaf_idx[i, t]
                if leaf in leaf_maps[t]:
                    col = offset + leaf_maps[t][leaf]
                    rows.append(i); cols.append(col); data.append(1.0)
            offset += len(leaf_maps[t])
        total_cols = sum(len(m) for m in leaf_maps)
        return sp.csr_matrix((data, (rows, cols)), shape=(leaf_idx.shape[0], total_cols))

    Phi_train = get_leaf_matrix(leaf_idx_train)
    Phi_test  = get_leaf_matrix(leaf_idx_test)

    W_sqrt = sp.diags(np.sqrt(hess_train))
    Phi_w  = W_sqrt @ Phi_train

    def Av(v):
        return (Phi_w.T @ (Phi_w @ v)) / n_train + damping * v

    print("Solving (H + gamma I)s = change L_j * THETA_j ...")
    ihvps = []
    for j in range(n_train):
        if j % 500 == 0: print(f" CG solve {j}/{n_train}")
        b = grad_train[j] * Phi_train[j].toarray().ravel()
        s = cg_solve(Av, b)
        ihvps.append(s)

    print("Computing influence matrix...")
    IF_matrix = np.zeros((n_test, n_train))
    for i in range(n_test):
        phi_i = Phi_test[i].toarray().ravel()
        for j in range(n_train):
            IF_matrix[i, j] = -grad_test[i] * np.dot(phi_i, ihvps[j])
        if i % 100 == 0: print(f"  Row {i}/{n_test}")
    return IF_matrix

# ------------------- 4. SHAP Interactions -----------------
def compute_shap_interactions(model, X_test):
    # Ensure data types are correct for SHAP
    X_test = np.array(X_test, dtype=np.float32)
    
    explainer = shap.TreeExplainer(model)
    shap_inter = explainer.shap_interaction_values(X_test)
    return np.array(shap_inter)

# ------------------- 5. Permutation Test -----------------
def permutation_test(A, B, n_permutations=10000):
    obs_diff = B.mean() - A.mean()
    combined = np.hstack([A.flatten(), B.flatten()])
    count = 0; len_A = A.size
    for _ in range(n_permutations):
        np.random.shuffle(combined)
        diff = combined[:len_A].mean() - combined[len_A:].mean()
        if diff >= obs_diff: count += 1
    p_val = (count + 1) / (n_permutations + 1)
    return obs_diff, p_val

# ------------------- 6. SHAP per Observation + Summary Plot -----------------
def save_shap_per_observation(model, X_test, feature_names, id_test, race_test, writer, sheet_prefix=''):
    print("Computing SHAP main effects …")
    X_test = np.array(X_test, dtype=np.float32)
    
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_test)
    shap_df = pd.DataFrame(shap_vals, columns=feature_names)
    
    # Add ID and Race as first columns
    shap_df.insert(0, 'Race_Test', race_test)
    shap_df.insert(0, 'Test_ID', id_test)
    
    shap_df.to_excel(writer, sheet_name=f'{sheet_prefix}SHAP_Main_Effects', index=False)

    print("Computing SHAP interaction values …")
    shap_inter = explainer.shap_interaction_values(X_test)
    for idx in range(min(5, len(X_test))):  # Limit to first 5 to avoid Excel limits
        inter_mat = shap_inter[idx]
        inter_df = pd.DataFrame(inter_mat, index=feature_names, columns=feature_names)
        sheet = f"{sheet_prefix}Inter_ID{id_test[idx]}_{race_test[idx]}"
        inter_df.to_excel(writer, sheet_name=sheet[:31])

    print("SHAP sheets written")

# ------------------- 6b. SHAP Summary Plot -----------------

def plot_shap_summary(model, X_test, feature_names, plot_dir, target_title, file_suffix=''):
    os.makedirs(plot_dir, exist_ok=True)
    print("Creating SHAP summary (beeswarm) plot …")
    X_test = np.array(X_test, dtype=np.float32)
    
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_test)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_test, feature_names=feature_names,
                      show=False, plot_type="dot", max_display=20)
    plt.title(f"SHAP Summary{file_suffix} - Feature Impact on Predicted {target_title}", fontsize=14)
    plt.tight_layout()
    out_path = os.path.join(plot_dir, f"shap_summary_beeswarm{file_suffix}.png")
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"->Saved: {out_path}")

# ------------------- 7. MAIN FUNCTION -----------------
def compas_xgb_full_analysis(df,
                             excel_path="results_compas.xlsx",
                             plot_dir="influence_histograms",
                             target_col='decile_binary',
                             label_name='DecileScore',
                             target_title='High Risk (Decile 6-10)'):
    os.makedirs(plot_dir, exist_ok=True)
    
    # Compute target
    if target_col == 'decile_binary':
        df['target'] = (df['decile_score'] >= 6).astype(int)
        extra_req = ['decile_score']
    else:
        df['target'] = df[target_col].astype(int)
        extra_req = [target_col]

    # === Drop rows with missing critical values ===
    required_cols = ['sex_original', 'race_original', 'charge_degree_original', 'risk_original',
                     'age', 'priors_count', 'juv_fel_count', 'juv_misd_count', 
                     'juv_other_count'] + extra_req
    df = df.dropna(subset=required_cols).reset_index(drop=True)

    print(f"Final dataset size: {len(df)}")
    print(f"Binary target distribution:\n{df['target'].value_counts()}")

    # Add intersectional group
    df['group'] = df['race_original'] + '_' + df['sex_original']

    # ---- Create proper feature set with dummy variables ----
    # Base features (numeric)
    base_features = ['age', 'priors_count', 'juv_fel_count', 'juv_misd_count', 'juv_other_count']
    
    # Create dummy variables for all categorical features
    sex_dummies = pd.get_dummies(df['sex_original'], prefix='Sex')
    race_dummies = pd.get_dummies(df['race_original'], prefix='Race')
    risk_dummies = pd.get_dummies(df['risk_original'], prefix='Risk')
    charge_dummies = pd.get_dummies(df['charge_degree_original'], prefix='Charge')
    
    # Combine all features
    X_df = pd.concat([
        df[base_features],
        sex_dummies,
        race_dummies,
        risk_dummies,
        charge_dummies
    ], axis=1)
    
    # Ensure all data is numeric
    for col in X_df.columns:
        X_df[col] = pd.to_numeric(X_df[col], errors='coerce')
    
    # Drop any rows with NaN values
    X_df = X_df.dropna()
    df = df.loc[X_df.index]  # Align the target and other columns
    
    feature_names = X_df.columns.tolist()
    print(f"Features used ({len(feature_names)}): {feature_names}")
    
    X_all = X_df.values.astype(np.float32)
    y_all = df['target'].values.astype(int)  # Using target as label
    race_all = df['race_original'].values
    sex_all = df['sex_original'].values
    group_all = df['group'].values
    ids_all = df['id'].values
    decile_all = df['decile_score'].values  # Keep original for reference

    print(f"Final dataset size after cleaning: {len(X_all)}")
    print(f"Target variable: {label_name} binary (0=Low, 1=High)")
    print(f"Class distribution: {np.bincount(y_all)}")

    # ---- splits --------------------------------------------------------
    train_idx, test_idx = train_test_split(range(len(X_all)), test_size=0.2, random_state=43, stratify=y_all)
    X_temp, X_test = X_all[train_idx], X_all[test_idx]
    y_temp, y_test = y_all[train_idx], y_all[test_idx]
    race_temp, race_test = race_all[train_idx], race_all[test_idx]
    sex_temp, sex_test = sex_all[train_idx], sex_all[test_idx]
    group_temp, group_test = group_all[train_idx], group_all[test_idx]
    id_temp, id_test = ids_all[train_idx], ids_all[test_idx]

    train_idx2, val_idx = train_test_split(range(len(X_temp)), test_size=0.2, random_state=43, stratify=y_temp)
    X_train, X_val = X_temp[train_idx2], X_temp[val_idx]
    y_train, y_val = y_temp[train_idx2], y_temp[val_idx]
    race_train, race_val = race_temp[train_idx2], race_temp[val_idx]
    sex_train, sex_val = sex_temp[train_idx2], sex_temp[val_idx]
    group_train, group_val = group_temp[train_idx2], group_temp[val_idx]
    id_train, id_val = id_temp[train_idx2], id_temp[val_idx]

    # ---- train ---------------------------------------------------------
    print("Training XGBoost...")
    model = train_xgb(X_train, y_train, X_val=X_val, y_val=y_val)

    # ---- model performance ---------------------------------------------
    from sklearn.metrics import accuracy_score, roc_auc_score
    dtrain = xgb.DMatrix(X_train); dtest = xgb.DMatrix(X_test)
    y_pred_proba = model.predict(dtest)
    y_pred = (y_pred_proba > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"Model Performance - Accuracy: {accuracy:.4f}, AUC: {auc:.4f}")

    # ---- Create predictions sheet with actual vs predicted by race ----
    print("Creating predictions summary by race...")
    
    # Get predictions for all test data
    test_predictions = []
    for i in range(len(X_test)):
        test_predictions.append({
            'ID': id_test[i],
            'Race': race_test[i],
            'Sex': sex_test[i],
            'Actual_Label': y_test[i],
            'Predicted_Probability': y_pred_proba[i],
            'Predicted_Label': y_pred[i],
            'Correct': y_test[i] == y_pred[i]
        })
    
    predictions_df = pd.DataFrame(test_predictions)
    
    # Also get predictions for training data for completeness
    y_train_pred_proba = model.predict(xgb.DMatrix(X_train))
    y_train_pred = (y_train_pred_proba > 0.5).astype(int)
    
    train_predictions = []
    for i in range(len(X_train)):
        train_predictions.append({
            'ID': id_train[i],
            'Race': race_train[i],
            'Sex': sex_train[i],
            'Actual_Label': y_train[i],
            'Predicted_Probability': y_train_pred_proba[i],
            'Predicted_Label': y_train_pred[i],
            'Correct': y_train[i] == y_train_pred[i],
            'Data_Split': 'Train'
        })
    
    # Add data split info to test predictions
    for i in range(len(predictions_df)):
        predictions_df.at[i, 'Data_Split'] = 'Test'
    
    # Combine train and test predictions
    all_predictions_df = pd.concat([
        pd.DataFrame(train_predictions),
        predictions_df
    ], ignore_index=True)

    # ---- influence -----------------------------------------------------
    print("\nComputing influence matrices...")
    IF_matrix = compute_influence_matrix_xgb(model, X_train, y_train, X_test, y_test, damping=1.0)
    IF_train_matrix = compute_influence_matrix_xgb(model, X_train, y_train, X_train, y_train, damping=1.0)

    # ---- Create improved influence dataframes with IDs and Race ----
    # For test vs train influence matrix
    test_data = []
    for i in range(len(test_idx)):
        test_data.append({
            'Test_ID': id_test[i],
            'Race_Test': race_test[i],
            'Sex_Test': sex_test[i]
        })
    test_info_df = pd.DataFrame(test_data)
    
    train_data = []
    for j in range(len(train_idx2)):
        train_data.append({
            'Train_ID': id_train[j],
            'Race_Train': race_train[j],
            'Sex_Train': sex_train[j]
        })
    train_info_df = pd.DataFrame(train_data)
    
    # Create the influence matrix with proper indexing
    df_IF_detailed = pd.DataFrame(IF_matrix)
    
    # Set column names as Train_ID_Race_Sex
    df_IF_detailed.columns = [f"ID{id_train[j]}_{race_train[j]}_{sex_train[j]}" for j in range(len(train_idx2))]
    
    # Add Test_ID, Race_Test and Sex_Test as first columns
    df_IF_detailed.insert(0, 'Sex_Test', sex_test)
    df_IF_detailed.insert(0, 'Race_Test', race_test)
    df_IF_detailed.insert(0, 'Test_ID', id_test)

    # Similarly for train vs train matrix
    df_IF_train_detailed = pd.DataFrame(IF_train_matrix)
    df_IF_train_detailed.columns = [f"ID{id_train[j]}_{race_train[j]}_{sex_train[j]}" for j in range(len(train_idx2))]
    df_IF_train_detailed.insert(0, 'Sex_Train', sex_train)
    df_IF_train_detailed.insert(0, 'Race_Train', race_train)
    df_IF_train_detailed.insert(0, 'Train_ID', id_train)

    # ---- cross-group means & histograms --------------------------------
    races = ['African-American', 'Caucasian', 'Hispanic', 'Other']
    cross_means = {}; influence_submats = {}
    print("\nComputing cross-group influence (race)...")
    for r1 in races:
        for r2 in races:
            if r1 == r2: continue
            idx_test = [i for i, r in enumerate(race_test) if r == r1]
            idx_train = [j for j, r in enumerate(race_train) if r == r2]
            if len(idx_test) < 5 or len(idx_train) < 5: continue
            sub = IF_matrix[np.ix_(idx_test, idx_train)]
            cross_means[(r1, r2)] = sub.mean()
            influence_submats[(r1, r2)] = sub.flatten()

    cross_df = pd.DataFrame(index=races, columns=races)
    for (r1, r2), val in cross_means.items():
        cross_df.loc[r1, r2] = val
    cross_df = cross_df.astype(float).fillna(0)

    # Intersectional cross-group
    groups = sorted(set(group_train) | set(group_test))
    cross_means_inter = {}; influence_submats_inter = {}
    print("\nComputing cross-group influence (intersectional)...")
    for g1 in groups:
        for g2 in groups:
            if g1 == g2: continue
            idx_test = [i for i, g in enumerate(group_test) if g == g1]
            idx_train = [j for j, g in enumerate(group_train) if g == g2]
            if len(idx_test) < 5 or len(idx_train) < 5: continue
            sub = IF_matrix[np.ix_(idx_test, idx_train)]
            cross_means_inter[(g1, g2)] = sub.mean()
            influence_submats_inter[(g1, g2)] = sub.flatten()

    cross_df_inter = pd.DataFrame(index=groups, columns=groups)
    for (g1, g2), val in cross_means_inter.items():
        cross_df_inter.loc[g1, g2] = val
    cross_df_inter = cross_df_inter.astype(float).fillna(0)

    # ---- SHAP analysis ---------------------------
    print("\nComputing SHAP values...")
    try:
        shap_inter = compute_shap_interactions(model, X_test)
        
        # Create SHAP summary for different feature types
        sex_features = [f for f in feature_names if f.startswith('Sex_')]
        race_features = [f for f in feature_names if f.startswith('Race_')]
        risk_features = [f for f in feature_names if f.startswith('Risk_')]
        charge_features = [f for f in feature_names if f.startswith('Charge_')]
        
        # Sex SHAP summary
        sex_shap_summary = pd.DataFrame(index=['Male', 'Female'], columns=['Mean_Abs_SHAP'])
        for sex_feature in sex_features:
            sex_name = sex_feature.replace('Sex_', '')
            feature_idx = feature_names.index(sex_feature)
            sex_shap_vals = shap_inter[:, feature_idx, feature_idx]  # Main effect
            sex_shap_summary.loc[sex_name, 'Mean_Abs_SHAP'] = np.mean(np.abs(sex_shap_vals))
        
        # Race SHAP summary
        race_shap_summary = pd.DataFrame(index=races, columns=['Mean_Abs_SHAP'])
        for race_feature in race_features:
            race_name = race_feature.replace('Race_', '')
            feature_idx = feature_names.index(race_feature)
            race_shap_vals = shap_inter[:, feature_idx, feature_idx]  # Main effect
            race_shap_summary.loc[race_name, 'Mean_Abs_SHAP'] = np.mean(np.abs(race_shap_vals))
        
        # Risk SHAP summary
        risk_levels = ['Low', 'Medium', 'High']
        risk_shap_summary = pd.DataFrame(index=risk_levels, columns=['Mean_Abs_SHAP'])
        for risk_feature in risk_features:
            risk_name = risk_feature.replace('Risk_', '')
            feature_idx = feature_names.index(risk_feature)
            risk_shap_vals = shap_inter[:, feature_idx, feature_idx]  # Main effect
            risk_shap_summary.loc[risk_name, 'Mean_Abs_SHAP'] = np.mean(np.abs(risk_shap_vals))
        
        # Charge degree SHAP summary
        charge_levels = ['Misdemeanor', 'Felony']
        charge_shap_summary = pd.DataFrame(index=charge_levels, columns=['Mean_Abs_SHAP'])
        for charge_feature in charge_features:
            charge_name = 'Misdemeanor' if charge_feature == 'Charge_M' else 'Felony'
            feature_idx = feature_names.index(charge_feature)
            charge_shap_vals = shap_inter[:, feature_idx, feature_idx]  # Main effect
            charge_shap_summary.loc[charge_name, 'Mean_Abs_SHAP'] = np.mean(np.abs(charge_shap_vals))
            
    except Exception as e:
        print(f"SHAP analysis failed: {e}")
        sex_shap_summary = pd.DataFrame()
        race_shap_summary = pd.DataFrame()
        risk_shap_summary = pd.DataFrame()
        charge_shap_summary = pd.DataFrame()
        shap_inter = None

    # ---- permutation tests ------------------------------------
    p_values = {}
    print("\nPermutation tests (10,000 shuffles) for race:")
    for r1 in races:
        for r2 in races:
            if r1 == r2: continue
            idx_test1 = [i for i, r in enumerate(race_test) if r == r1]
            idx_test2 = [i for i, r in enumerate(race_test) if r == r2]
            idx_train1 = [j for j, r in enumerate(race_train) if r == r1]
            idx_train2 = [j for j, r in enumerate(race_train) if r == r2]
            if len(idx_test1) < 5 or len(idx_test2) < 5: continue
            A = IF_matrix[np.ix_(idx_test1, idx_train2)]
            B = IF_matrix[np.ix_(idx_test2, idx_train1)]
            if A.size == 0 or B.size == 0: continue
            diff, p = permutation_test(A, B)
            p_values[(r1, r2)] = p
            print(f"  {r1[:3]}->{r2[:3]} vs {r2[:3]}->{r1[:3]}: diff={diff:.6f}, p={p:.4f}")

    pval_df = pd.DataFrame(index=races, columns=races)
    for (r1, r2), p in p_values.items():
        pval_df.loc[r1, r2] = p
    pval_df = pval_df.astype(float).fillna(1.0)

    # Intersectional permutation
    p_values_inter = {}
    print("\nPermutation tests (10,000 shuffles) for intersectional:")
    for g1 in groups:
        for g2 in groups:
            if g1 == g2: continue
            idx_test1 = [i for i, g in enumerate(group_test) if g == g1]
            idx_test2 = [i for i, g in enumerate(group_test) if g == g2]
            idx_train1 = [j for j, g in enumerate(group_train) if g == g1]
            idx_train2 = [j for j, g in enumerate(group_train) if g == g2]
            if len(idx_test1) < 5 or len(idx_test2) < 5: continue
            A = IF_matrix[np.ix_(idx_test1, idx_train2)]
            B = IF_matrix[np.ix_(idx_test2, idx_train1)]
            if A.size == 0 or B.size == 0: continue
            diff, p = permutation_test(A, B)
            p_values_inter[(g1, g2)] = p
            print(f"  {g1} -> {g2} vs {g2} -> {g1}: diff={diff:.6f}, p={p:.4f}")

    pval_df_inter = pd.DataFrame(index=groups, columns=groups)
    for (g1, g2), p in p_values_inter.items():
        pval_df_inter.loc[g1, g2] = p
    pval_df_inter = pval_df_inter.astype(float).fillna(1.0)

    # ---- top-10 AA to C ------------------------------------------------
    aa_test_idx = [i for i, r in enumerate(race_test) if r == 'African-American']
    c_train_idx = [j for j, r in enumerate(race_train) if r == 'Caucasian']
    top10_df = pd.DataFrame()
    if aa_test_idx and c_train_idx:
        sub = IF_matrix[np.ix_(aa_test_idx, c_train_idx)]
        top = np.unravel_index(np.argsort(sub.ravel())[-10:][::-1], sub.shape)
        rows = []
        for k in range(10):
            i, j = top[0][k], top[1][k]
            rows.append({
                'Test_ID': id_test[aa_test_idx[i]],
                'Test_Race': race_test[aa_test_idx[i]],
                'Test_Sex': sex_test[aa_test_idx[i]],
                'Train_ID': id_train[c_train_idx[j]],
                'Train_Race': race_train[c_train_idx[j]],
                'Train_Sex': sex_train[c_train_idx[j]],
                'Influence': sub[i, j]
            })
        top10_df = pd.DataFrame(rows)

    # ---- fairness metrics ---------------------------------------------
    print("\nComputing fairness metrics...")
    def compute_fairness_metrics(y_true, y_pred, y_pred_proba, groups):
        unique_groups = np.unique(groups)
        results = []
        for g in unique_groups:
            mask = groups == g
            acc = accuracy_score(y_true[mask], y_pred[mask])
            auc = roc_auc_score(y_true[mask], y_pred_proba[mask]) if len(np.unique(y_true[mask])) > 1 else np.nan
            prec = precision_score(y_true[mask], y_pred[mask], zero_division=0)
            rec = recall_score(y_true[mask], y_pred[mask], zero_division=0)
            f1 = f1_score(y_true[mask], y_pred[mask], zero_division=0)
            # DP: P(hatY=1 | g)
            dp = y_pred[mask].mean()
            # EO: TPR, FPR
            mask1 = mask & (y_true == 1)
            tpr = y_pred[mask1].mean() if mask1.sum() > 0 else np.nan
            mask0 = mask & (y_true == 0)
            fpr = y_pred[mask0].mean() if mask0.sum() > 0 else np.nan
            # PP: PPV
            mask_hat1 = mask & (y_pred == 1)
            ppv = y_true[mask_hat1].mean() if mask_hat1.sum() > 0 else np.nan
            results.append({
                'Group': g,
                'Count': mask.sum(),
                'Accuracy': acc,
                'AUC': auc,
                'Precision': prec,
                'Recall/TPR': rec,
                'F1': f1,
                'DP (P(hat=1))': dp,
                'FPR': fpr,
                'PPV': ppv
            })
        return pd.DataFrame(results)

    fairness_df = compute_fairness_metrics(y_test, y_pred, y_pred_proba, race_test)
    fairness_inter_df = compute_fairness_metrics(y_test, y_pred, y_pred_proba, group_test)

    # ---- influence decomposition for fairness and performance ----
    print("\nDecomposing performance and fairness with influences...")
    self_means = {}
    for r in races:
        idx_train_r = [j for j, rr in enumerate(race_train) if rr == r]
        idx_test_r = [i for i, rr in enumerate(race_test) if rr == r]
        if len(idx_test_r) < 1 or len(idx_train_r) < 1: continue
        sub_self = IF_matrix[np.ix_(idx_test_r, idx_train_r)]
        self_means[r] = sub_self.mean()

    fairness_df['Self_Influence'] = fairness_df['Group'].map(self_means).fillna(0)
    fairness_df['Avg_Between_Influence'] = [cross_df.loc[r, cross_df.columns != r].mean() if r in cross_df.index else np.nan for r in fairness_df['Group']]

    # Correlations with self/between
    if len(races) > 1 and len(self_means) > 1:
        metrics = ['Recall/TPR', 'FPR', 'Accuracy', 'PPV']
        for met in metrics:
            vals = fairness_df[met].values
            self_corr = np.corrcoef(fairness_df['Self_Influence'], vals)[0,1]
            between_corr = np.corrcoef(fairness_df['Avg_Between_Influence'], vals)[0,1]
            print(f"Correlation Self with {met}: {self_corr:.4f}")
            print(f"Correlation Avg_Between with {met}: {between_corr:.4f}")

    # Pairwise decompositions for fairness violations
    decomp_rows = []
    for ii in range(len(races)):
        for jj in range(ii+1, len(races)):
            r1 = races[ii]
            r2 = races[jj]
            row1 = fairness_df[fairness_df['Group'] == r1].iloc[0]
            row2 = fairness_df[fairness_df['Group'] == r2].iloc[0]
            dp_diff = row1['DP (P(hat=1))'] - row2['DP (P(hat=1))']
            tpr_diff = row1['Recall/TPR'] - row2['Recall/TPR']
            fpr_diff = row1['FPR'] - row2['FPR']
            ppv_diff = row1['PPV'] - row2['PPV']
            between_asym = cross_df.loc[r1, r2] - cross_df.loc[r2, r1] if r1 in cross_df.index and r2 in cross_df.columns else np.nan
            self_diff = row1['Self_Influence'] - row2['Self_Influence']
            decomp_rows.append({
                'Pair': f"{r1} vs {r2}",
                'DP_Diff': dp_diff,
                'Between_Asym': between_asym,
                'TPR_Diff': tpr_diff,
                'FPR_Diff': fpr_diff,
                'PPV_Diff': ppv_diff,
                'Self_Diff': self_diff
            })
    decomp_df = pd.DataFrame(decomp_rows)

    # ---- mitigation ----------------------------------------------------
    print("\nPerforming influence-based mitigation (downweight top harmful Caucasian to African-American)...")
    weights = np.ones(len(y_train))
    if aa_test_idx and c_train_idx:
        sub = IF_matrix[np.ix_(aa_test_idx, c_train_idx)]
        flat_idx = np.argsort(sub.ravel())[-10:][::-1]  # top 10 highest (assuming positive harmful)
        top_i, top_j = np.unravel_index(flat_idx, sub.shape)
        harmful_train_idx = [c_train_idx[tj] for tj in top_j]
        for idx in harmful_train_idx:
            weights[idx] = 0.1  # downweight
    model_mit = train_xgb(X_train, y_train, X_val, y_val, weights=weights)
    y_pred_proba_mit = model_mit.predict(dtest)
    y_pred_mit = (y_pred_proba_mit > 0.5).astype(int)
    acc_mit = accuracy_score(y_test, y_pred_mit)
    auc_mit = roc_auc_score(y_test, y_pred_proba_mit)
    print(f"Mitigated Model Performance - Accuracy: {acc_mit:.4f}, AUC: {auc_mit:.4f}")
    perf_mit_df = pd.DataFrame({
        'Metric': ['Accuracy', 'AUC-ROC'],
        'Value': [acc_mit, auc_mit]
    })
    fairness_mit_df = compute_fairness_metrics(y_test, y_pred_mit, y_pred_proba_mit, race_test)
    fairness_mit_inter_df = compute_fairness_metrics(y_test, y_pred_mit, y_pred_proba_mit, group_test)

    # SHAP for mitigated
    try:
        shap_inter_mit = compute_shap_interactions(model_mit, X_test)
        # Example summary for race
        race_shap_summary_mit = pd.DataFrame(index=races, columns=['Mean_Abs_SHAP'])
        for race_feature in race_features:
            race_name = race_feature.replace('Race_', '')
            feature_idx = feature_names.index(race_feature)
            race_shap_vals = shap_inter_mit[:, feature_idx, feature_idx]  # Main effect
            race_shap_summary_mit.loc[race_name, 'Mean_Abs_SHAP'] = np.mean(np.abs(race_shap_vals))
    except Exception as e:
        print(f"Mitigated SHAP failed: {e}")
        shap_inter_mit = None
        race_shap_summary_mit = pd.DataFrame()

    # ------------------- EXCEL EXPORT (includes SHAP) -------------------
    print(f"\nSaving all results to '{excel_path}' …")
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Save predictions sheet
        all_predictions_df.to_excel(writer, sheet_name='Predictions_By_Race', index=False)
        
        # Save detailed influence matrices with IDs
        df_IF_detailed.to_excel(writer, sheet_name='Influence_Test_vs_Train', index=False)
        df_IF_train_detailed.to_excel(writer, sheet_name='Influence_Train_vs_Train', index=False)
        
        cross_df.to_excel(writer, sheet_name='CrossGroup_Means_Race')
        cross_df_inter.to_excel(writer, sheet_name='CrossGroup_Means_Inter')
        if not sex_shap_summary.empty:
            sex_shap_summary.to_excel(writer, sheet_name='SHAP_Sex_Summary')
        if not race_shap_summary.empty:
            race_shap_summary.to_excel(writer, sheet_name='SHAP_Race_Summary')
        if not risk_shap_summary.empty:
            risk_shap_summary.to_excel(writer, sheet_name='SHAP_Risk_Summary')
        if not charge_shap_summary.empty:
            charge_shap_summary.to_excel(writer, sheet_name='SHAP_Charge_Summary')
        pval_df.to_excel(writer, sheet_name='Permutation_Pvalues_Race')
        pval_df_inter.to_excel(writer, sheet_name='Permutation_Pvalues_Inter')
        top10_df.to_excel(writer, sheet_name='Top10_AA_to_C', index=False)
        
        # Add SHAP main effects if available
        if shap_inter is not None:
            try:
                save_shap_per_observation(model, X_test, feature_names, id_test, race_test, writer)
            except Exception as e:
                print(f"Could not save detailed SHAP observations: {e}")
        
        # Add model performance sheet
        perf_df = pd.DataFrame({
            'Metric': ['Accuracy', 'AUC-ROC'],
            'Value': [accuracy, auc]
        })
        perf_df.to_excel(writer, sheet_name='Model_Performance', index=False)
        
        # Add feature list sheet
        feature_info = pd.DataFrame({
            'Feature_Name': feature_names,
            'Type': ['Numeric' if f in base_features else 'Categorical_Dummy' for f in feature_names]
        })
        feature_info.to_excel(writer, sheet_name='Feature_Info', index=False)

        # Fairness
        fairness_df.to_excel(writer, sheet_name='Fairness_Metrics', index=False)
        fairness_inter_df.to_excel(writer, sheet_name='Fairness_Metrics_Inter', index=False)
        decomp_df.to_excel(writer, sheet_name='Fairness_Decomposition', index=False)

        # Mitigated
        perf_mit_df.to_excel(writer, sheet_name='Mitigated_Performance', index=False)
        fairness_mit_df.to_excel(writer, sheet_name='Mitigated_Fairness', index=False)
        fairness_mit_inter_df.to_excel(writer, sheet_name='Mitigated_Fairness_Inter', index=False)
        if not race_shap_summary_mit.empty:
            race_shap_summary_mit.to_excel(writer, sheet_name='Mitigated_SHAP_Race')
        if shap_inter_mit is not None:
            try:
                save_shap_per_observation(model_mit, X_test, feature_names, id_test, race_test, writer, sheet_prefix='Mitigated_')
            except Exception as e:
                print(f"Could not save mitigated SHAP: {e}")
        
    print(f"Excel saved: {excel_path}")

    # ------------------- HISTOGRAM PLOTS --------------------------------
       ###########################################################################

    print(f"\nPlotting influence histograms -> '{plot_dir}/'")
    colors = {'African-American': '#d62728', 'Caucasian': '#1f77b4',
              'Hispanic': '#2ca02c', 'Other': '#7f7f7f'}
    for (r1, r2), values in influence_submats.items():
        if len(values) == 0: continue
        plt.figure(figsize=(9, 5.5))
        sns.histplot(values, bins=60, kde=True, color=colors.get(r2, 'gray'), alpha=0.75, stat='density')
        mean_val = values.mean(); std_val = values.std()
        plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_val:.2e}')
        plt.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1.5)
        plt.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1.5)
        plt.title(f'Influence: {r2} to {r1}\nN = {len(values):,}', fontsize=14)
        plt.xlabel('Influence Value'); plt.ylabel('Density'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        safe_r1 = r1.replace(' ', '_'); safe_r2 = r2.replace(' ', '_')
        plt.savefig(f"{plot_dir}/hist_influence_{safe_r2}_to_{safe_r1}.png", dpi=200, bbox_inches='tight')
        plt.close()

    # Intersectional histograms
    colors_inter = {}
    for g in groups:
        r_base = g.split('_')[0]
        colors_inter[g] = colors.get(r_base, 'gray')
    for (g1, g2), values in influence_submats_inter.items():
        if len(values) == 0: continue
        plt.figure(figsize=(9, 5.5))
        sns.histplot(values, bins=60, kde=True, color=colors_inter.get(g2, 'gray'), alpha=0.75, stat='density')
        mean_val = values.mean(); std_val = values.std()
        plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_val:.2e}')
        plt.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1.5)
        plt.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1.5)
        plt.title(f'Influence: {g2} to {g1}\nN = {len(values):,}', fontsize=14)
        plt.xlabel('Influence Value'); plt.ylabel('Density'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        safe_g1 = g1.replace(' ', '_').replace('-', '_'); safe_g2 = g2.replace(' ', '_').replace('-', '_')
        plt.savefig(f"{plot_dir}/hist_influence_inter_{safe_g2}_to_{safe_g1}.png", dpi=200, bbox_inches='tight')
        plt.close()
    print(f"All {len(influence_submats) + len(influence_submats_inter)} histograms saved")

    # ------------------- SHAP SUMMARY PLOT -----------------------------
    if shap_inter is not None:
        try:
            plot_shap_summary(model, X_test, feature_names, plot_dir, target_title)
        except Exception as e:
            print(f"Could not create SHAP summary plot: {e}")

    # Mitigated SHAP plot
    plot_shap_summary(model_mit, X_test, feature_names, plot_dir, target_title, file_suffix='_Mitigated')

    return IF_matrix, df_IF_detailed, df_IF_train_detailed, cross_means, p_values

# --------------------------------------------------------------
#  RUN THE EXPERIMENT
# --------------------------------------------------------------
if __name__ == "__main__":
    np.random.seed(43)
    
    # === Convert decile_score to binary target ===
    # 1-5 = 0 (Low Risk), 6-10 = 1 (High Risk)
    df['decile_binary'] = (df['decile_score'] >= 6).astype(int)
    print(f"Decile score distribution: {df['decile_score'].value_counts().sort_index()}")
    print(f"Binary target distribution:\n{df['decile_binary'].value_counts()}")

    print("Starting COMPAS analysis with enhanced features...")
    print(f"Dataset size: {len(df)}")
    
    IF_matrix, df_IF_detailed, df_IF_train_detailed, cross_means, p_values = compas_xgb_full_analysis(
        df,
        excel_path="results_compas_DecileScore.xlsx",
        plot_dir="influence_histograms_DecileScore",
        target_col='decile_binary',
        label_name='DecileScore',
        target_title='High Risk (Decile 6-10)'
    )

    IF_matrix, df_IF_detailed, df_IF_train_detailed, cross_means, p_values = compas_xgb_full_analysis(
        df,
        excel_path="results_compas_Recidivism.xlsx",
        plot_dir="influence_histograms_Recidivism",
        target_col='is_recid',
        label_name='Recidivism',
        target_title='Recidivism'
    )
    
    print("\n=== Experiment Completed ===")

OSError: [WinError 126] The specified module could not be found. Error loading "C:\Users\ojoak\AppData\Roaming\Python\Python312\site-packages\torch\lib\torch_python.dll" or one of its dependencies.